# 09 — Advanced: A2A Client

**Stage 9 of the workshop (Production, extended).** Discovers an agent's capabilities from its published agent card, then sends it a message over HTTP — client half of a two-process pair.

## Problem

Getting from "works on my laptop" to something a team can rely on — consuming a remotely deployed agent as a standalone service, over a standard protocol, rather than calling a function in the same process.

## Concept

A2A differs from MCP in what crosses the wire: MCP exposes *tools* to an agent; A2A exposes a *whole agent* to other agents/services as a standalone deployment. This client doesn't construct a Strands `Agent` itself — it's a generic A2A client (`A2ACardResolver`, `ClientFactory`) that discovers what the remote agent can do from its published "agent card" and then talks to it purely over HTTP.

## Architecture

```
A2ACardResolver(httpx_client, "http://127.0.0.1:9000")
        │
        ▼
  card = resolver.get_agent_card()      discover name/description
        │
        ▼
  client = ClientFactory(config).create(card)
        │
        ▼
  client.send_message("What is 125 plus 375?")
        │
        ▼
  async for event in ... :
     task.artifacts[-1].parts[-1].root.text   ── extract answer text
        │
        ▼
  print("Agent answered: ...")
```

## Before running this notebook

This is the CLIENT half. `4-a2a_server.py` must already be running in a **separate terminal/notebook**, listening on `http://127.0.0.1:9000`, before you run the cells below — this notebook does not start the server itself.

```
cd workshop/09-advanced
uv run 4-a2a_server.py
```

## Step 1 — Imports and target URL

In [1]:

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory, create_text_message_object

AGENT_URL = "http://127.0.0.1:9000"


## Step 2 — Define the discovery + send flow

In [2]:
async def main() -> None:
    async with httpx.AsyncClient(timeout=60.0) as httpx_client:
        resolver = A2ACardResolver(httpx_client, AGENT_URL)
        card = await resolver.get_agent_card()
        print(f"Discovered agent: {card.name} — {card.description}")

        config = ClientConfig(httpx_client=httpx_client, streaming=False)
        client = ClientFactory(config).create(card)
        message = create_text_message_object(content="What is 125 plus 375?")

        async for event in client.send_message(message):
            task, _update = event if isinstance(event, tuple) else (event, None)
            if hasattr(task, "artifacts") and task.artifacts:
                text = task.artifacts[-1].parts[-1].root.text
                print(f"Agent answered: {text.strip()}")
            else:
                print(event)


## Step 3 — Run it

Only run this after `4-a2a_server.py` is already up and listening on port 9000 in another process.

In [3]:
# Jupyter kernels already run inside an asyncio event loop, so
# asyncio.run() (correct for the plain .py script) raises
# "asyncio.run() cannot be called from a running event loop" here.
# Use top-level await instead — supported directly in notebook cells.
await main()

Discovered agent: calculator_agent — Answers arithmetic questions.


Agent answered: 500
